# CV4CHL — Notebook 02: XGB baseline (view-aware feature selection + clinical features)

**Platform**: Colab (CPU) or local
**Runtime**: ~35 min CPU
**Pre-requisites**: `1_data/processed/keypoints.pkl`, `1_data/processed/train_ready.pkl`

This notebook trains the XGB baseline used to generate the Track-1 baseline
predictions consumed by the rest of the pipeline. Methodology:

1. **View-aware feature SELECTION**: coronal items (4, 5, 13, 14, 15) use
   `cor_*` + `all_*` + `n_*`; sagittal items use `sag_*` + `all_*` + `n_*`.
2. **Clinical features (extracted inline from keypoints)**: pelvic drop (Trendelenburg),
   knee recurvatum (hyperextension), cadence regularity.
3. **L-R asymmetry features**: per-feature `|left - right|` for hemiplegic patterns.
4. **LOSO CV** over 4 strategies × 3 models (XGBoost / LightGBM / CatBoost).
5. **Per-item config selection via margin gating** (switch only if margin >= 0.03 over baseline).
6. **Items 4 + 13 view-aware override** (Cell 8b) — items 4 (HindfootVarusValgus)
   and 13 (HipFlexExtSwing) are coronal-plane phenomena better captured by
   view-aware coronal feature subsets than by the all-feature baseline.
7. Output canonical artifacts: `xgb_baseline.csv` + `reference_baseline.csv`.

Outputs land under `5_outputs/submissions/` rooted at `CV4CHL_REPO_ROOT` (set by
`reproduce.py`) or, when run standalone in Colab, at `/content/drive/MyDrive/CVPR_CH4CHL`.


In [ ]:
# === Cell 1: Setup & Imports ===
import time, os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from scipy.signal import butter, filtfilt, find_peaks
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

_cell_times = {}
_cell_start = None
def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'▶ {name}')
def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'✓ {name} — {elapsed:.1f}s ({elapsed/60:.1f}min)')

NOTEBOOK_NAME = 'CV4CHL_02_train_xgb_baseline'

try:
    import xgboost as xgb
except ImportError:
    os.system('pip install -q xgboost'); import xgboost as xgb
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip install -q lightgbm'); import lightgbm as lgb

HAVE_CATBOOST = False
try:
    from catboost import CatBoostClassifier
    import catboost
    HAVE_CATBOOST = True
except (ImportError, ValueError) as _e:
    print(f'WARNING: catboost unavailable ({type(_e).__name__}: {str(_e)[:80]}); LOSO CV will use XGBoost + LightGBM only.')
    CatBoostClassifier = None

from sklearn.metrics import accuracy_score
print(f'Setup complete. HAVE_CATBOOST={HAVE_CATBOOST}')


In [ ]:
# === Cell 2: Config & Load Data ===
cell_start('Cell 2: Config & Load Data')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
else:
    ROOT = os.environ.get('CV4CHL_REPO_ROOT', os.getcwd())

PROC_DIR = f'{ROOT}/1_data/processed'
SUBMIT_DIR = f'{ROOT}/5_outputs/submissions'
os.makedirs(SUBMIT_DIR, exist_ok=True)

# Load existing handcrafted features
print('Loading train_ready.pkl...')
with open(f'{PROC_DIR}/train_ready.pkl', 'rb') as f:
    tr_data = pickle.load(f)

print(f'Keys: {list(tr_data.keys())}')
df_t1_train = tr_data['track1_train']  # training features + labels
df_t1_test = tr_data['track1_test']    # test features
df_all = tr_data['all_features']       # all patient features (no labels)

print(f'Track1 train: {df_t1_train.shape}')
print(f'Track1 test: {df_t1_test.shape}')
print(f'All features: {df_all.shape}')

# Feature columns
FEAT_COLS_BASE = sorted([c for c in df_t1_train.columns
    if c.startswith(('sag_', 'cor_', 'all_', 'n_'))])
print(f'Base feature count: {len(FEAT_COLS_BASE)}')

# Identify feature groups
SAG_COLS = [c for c in FEAT_COLS_BASE if c.startswith('sag_')]
COR_COLS = [c for c in FEAT_COLS_BASE if c.startswith('cor_')]
ALL_COLS = [c for c in FEAT_COLS_BASE if c.startswith('all_')]
META_COLS = [c for c in FEAT_COLS_BASE if c.startswith('n_')]

print(f'  sag_*: {len(SAG_COLS)}, cor_*: {len(COR_COLS)}, all_*: {len(ALL_COLS)}, n_*: {len(META_COLS)}')

# EVGS labels
EVGS_ITEMS = [f'evgs_{i}' for i in range(1, 18)]

# Load keypoints for new feature computation
print('\nLoading keypoints.pkl...')
with open(f'{PROC_DIR}/keypoints.pkl', 'rb') as f:
    kp_data = pickle.load(f)

# Reorganize keypoints: (pid, vname) -> data
sample_key = list(kp_data.keys())[0]
if isinstance(sample_key, tuple):
    patient_kps = defaultdict(dict)
    for (pid, vname), vdata in kp_data.items():
        patient_kps[pid][vname] = vdata
    patient_kps = dict(patient_kps)
else:
    patient_kps = kp_data

print(f'Keypoints: {len(patient_kps)} patients')

# Test IDs
T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

# Keypoint indices
KP = {
    'nose': 0, 'left_shoulder': 5, 'right_shoulder': 6,
    'left_hip': 11, 'right_hip': 12, 'left_knee': 13, 'right_knee': 14,
    'left_ankle': 15, 'right_ankle': 16,
    'left_big_toe': 17, 'left_small_toe': 18, 'left_heel': 19,
    'right_big_toe': 20, 'right_small_toe': 21, 'right_heel': 22,
}

cell_end('Cell 2: Config & Load Data')


In [ ]:
# === Cell 3: Compute New Clinical Features ===
cell_start('Cell 3: Compute New Clinical Features')

def butterworth_lowpass(signal, cutoff_hz=6.0, fs=30.0, order=4):
    """Butterworth low-pass filter; fps=30 matches the preprocessing notebook."""
    if len(signal) < 3 * order:
        return signal.copy()
    nyquist = fs / 2.0
    normalized_cutoff = min(cutoff_hz / nyquist, 0.99)
    b, a = butter(order, normalized_cutoff, btype='low')
    try:
        return filtfilt(b, a, signal, padtype='odd',
                       padlen=min(3 * max(len(b), len(a)), len(signal) - 1))
    except Exception:
        return signal.copy()

def smooth_kps(kps, fs=30.0):
    T, N, D = kps.shape
    out = kps.copy()
    for j in range(N):
        for d in range(D):
            out[:, j, d] = butterworth_lowpass(kps[:, j, d], 6.0, fs)
    return out

def detect_gait_events(kps_smooth, side='left', fps=30.0):
    """Zeni method, fps=30 matched to preprocessing notebook."""
    T = len(kps_smooth)
    if T < 30:
        return None
    sacrum = (kps_smooth[:, KP['left_hip']] + kps_smooth[:, KP['right_hip']]) / 2.0
    heel_kp = KP[f'{side}_heel']
    hs_signal = kps_smooth[:, heel_kp, 0] - sacrum[:, 0]
    signal_range = np.ptp(hs_signal)
    if signal_range < 5.0:
        return None
    min_dist = max(int(fps * 0.4), 10)
    hs_prom = max(signal_range * 0.15, 3.0)
    hs_frames, _ = find_peaks(hs_signal, distance=min_dist, prominence=hs_prom)
    toe_kp = KP[f'{side}_big_toe']
    to_signal = kps_smooth[:, toe_kp, 0] - sacrum[:, 0]
    to_prom = max(np.ptp(to_signal) * 0.15, 3.0)
    to_frames, _ = find_peaks(-to_signal, distance=min_dist, prominence=to_prom)
    if len(hs_frames) < 2:
        return None
    return {'hs': hs_frames, 'to': to_frames}

def extract_new_clinical_features(pid, videos, side):
    """Compute NEW clinical features for one patient-side.
    Returns dict of ~8 new features.
    """
    feat = {}
    # Collect data from sagittal views
    sag_videos = [(vn, vd) for vn, vd in videos.items()
                  if isinstance(vd, dict) and vd.get('view') in ('left', 'right')]

    # Collect data from coronal views
    cor_videos = [(vn, vd) for vn, vd in videos.items()
                  if isinstance(vd, dict) and vd.get('view') in ('forward', 'backward')]

    # --- 1. Pelvic Drop (Trendelenburg) from CORONAL views ---
    # During single-leg stance, contralateral pelvis drops if abductors are weak
    pelvic_drops = []
    for vn, vd in cor_videos:
        kps = vd.get('keypoints')
        if kps is None or len(kps) < 20:
            continue
        if kps.shape[1] > 23:
            kps = kps[:, :23, :2].astype(np.float32)
        else:
            kps = kps[..., :2].astype(np.float32)
        kps_s = smooth_kps(kps)
        # Pelvic drop = max(|left_hip_y - right_hip_y|) during gait
        lh_y = kps_s[:, KP['left_hip'], 1]
        rh_y = kps_s[:, KP['right_hip'], 1]
        drop = np.abs(lh_y - rh_y)
        pelvic_drops.extend(drop.tolist())

    feat['new_pelvic_drop_mean'] = float(np.mean(pelvic_drops)) if pelvic_drops else 0.0
    feat['new_pelvic_drop_max'] = float(np.max(pelvic_drops)) if pelvic_drops else 0.0
    feat['new_pelvic_drop_std'] = float(np.std(pelvic_drops)) if len(pelvic_drops) > 1 else 0.0

    # --- 2. Knee Recurvatum (hyperextension) from SAGITTAL views ---
    knee_angles_all = []
    stride_times = []

    for vn, vd in sag_videos:
        kps = vd.get('keypoints')
        if kps is None or len(kps) < 20:
            continue
        if kps.shape[1] > 23:
            kps = kps[:, :23, :2].astype(np.float32)
        else:
            kps = kps[..., :2].astype(np.float32)
        kps_s = smooth_kps(kps)
        T = len(kps_s)

        # Knee angle
        hip_idx = KP[f'{side}_hip']
        knee_idx = KP[f'{side}_knee']
        ankle_idx = KP[f'{side}_ankle']

        for t in range(T):
            k = kps_s[t]
            ba = k[hip_idx] - k[knee_idx]
            bc = k[ankle_idx] - k[knee_idx]
            cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
            angle = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))
            knee_angles_all.append(angle)

        # Gait events for cadence
        events = detect_gait_events(kps_s, side=side, fps=30.0)
        if events is not None and len(events['hs']) >= 2:
            hs = events['hs']
            for i in range(len(hs) - 1):
                cycle_len = hs[i+1] - hs[i]
                if 15 <= cycle_len <= 90:  # valid at 30fps
                    stride_times.append(float(cycle_len / 30.0))

    # Knee recurvatum: frames where knee angle > 180 (hyperextension)
    # In 2D, angles > ~175 suggest hyperextension
    if knee_angles_all:
        ka = np.array(knee_angles_all)
        feat['new_knee_recurvatum_pct'] = float(np.mean(ka > 175))
        feat['new_knee_max_extension'] = float(np.max(ka))
    else:
        feat['new_knee_recurvatum_pct'] = 0.0
        feat['new_knee_max_extension'] = 0.0

    # --- 3. Cadence Regularity ---
    if len(stride_times) >= 2:
        st = np.array(stride_times)
        feat['new_cadence_mean'] = float(60.0 / np.mean(st))  # steps per minute
        feat['new_cadence_cv'] = float(np.std(st) / np.mean(st))  # regularity
    else:
        feat['new_cadence_mean'] = 0.0
        feat['new_cadence_cv'] = 0.0

    return feat

# Compute for all patients
new_feat_rows = []
all_pids = sorted(patient_kps.keys())

for pid in tqdm(all_pids, desc='Computing clinical features'):
    videos = patient_kps[pid]
    for side in ['left', 'right']:
        feat = extract_new_clinical_features(pid, videos, side)
        feat['patient_id'] = pid
        feat['side'] = side
        new_feat_rows.append(feat)

df_new = pd.DataFrame(new_feat_rows)
NEW_FEAT_COLS = [c for c in df_new.columns if c.startswith('new_')]
print(f'New clinical features: {len(NEW_FEAT_COLS)}')
print(f'  {NEW_FEAT_COLS}')
print(f'  Shape: {df_new.shape}')

# Quick stats
for col in NEW_FEAT_COLS:
    vals = df_new[col]
    print(f'  {col}: mean={vals.mean():.3f}, std={vals.std():.3f}, min={vals.min():.3f}, max={vals.max():.3f}')

cell_end('Cell 3: Compute New Clinical Features')


In [ ]:
# === Cell 4: Compute L-R Asymmetry Features ===
cell_start('Cell 4: L-R Asymmetry Features')

# For each patient, compute |left_feature - right_feature| for key angles
# This captures hemiplegic patterns

# Get all patient IDs from train
all_train_pids = df_t1_train['patient_id'].unique()
all_test_pids = df_t1_test['patient_id'].unique()
all_pids_feat = np.union1d(all_train_pids, all_test_pids)

# Key sagittal features to compute asymmetry
ASYM_SOURCE = [c for c in SAG_COLS if c.endswith(('_mean', '_range', '_min', '_max'))
               and not c.endswith('_std')]

asym_rows = []
for pid in tqdm(all_pids_feat, desc='Computing asymmetry'):
    # Get left and right feature rows
    left_row = df_all[(df_all['patient_id'] == pid) & (df_all['side'] == 'left')]
    right_row = df_all[(df_all['patient_id'] == pid) & (df_all['side'] == 'right')]

    if len(left_row) == 0 or len(right_row) == 0:
        continue

    for side in ['left', 'right']:
        row = {'patient_id': pid, 'side': side}
        for col in ASYM_SOURCE:
            lval = float(left_row[col].values[0]) if col in left_row.columns else 0.0
            rval = float(right_row[col].values[0]) if col in right_row.columns else 0.0
            row[f'asym_{col}'] = abs(lval - rval)
        asym_rows.append(row)

df_asym = pd.DataFrame(asym_rows)
ASYM_COLS = [c for c in df_asym.columns if c.startswith('asym_')]
print(f'Asymmetry features: {len(ASYM_COLS)}')

# Reduce: keep only top features by variance (avoid too many)
if len(ASYM_COLS) > 10:
    variances = df_asym[ASYM_COLS].var().sort_values(ascending=False)
    TOP_ASYM = list(variances.head(8).index)
    print(f'Top 8 by variance: {TOP_ASYM}')
else:
    TOP_ASYM = ASYM_COLS

cell_end('Cell 4: L-R Asymmetry Features')


In [ ]:
# === Cell 5: Merge Features & Define Groups ===
cell_start('Cell 5: Merge Features')

# Merge new clinical features into train and test
df_t1_train_ext = df_t1_train.merge(df_new[['patient_id', 'side'] + NEW_FEAT_COLS],
                                    on=['patient_id', 'side'], how='left')
df_t1_test_ext = df_t1_test.merge(df_new[['patient_id', 'side'] + NEW_FEAT_COLS],
                                  on=['patient_id', 'side'], how='left')

# Merge asymmetry features
df_t1_train_ext = df_t1_train_ext.merge(df_asym[['patient_id', 'side'] + TOP_ASYM],
                                        on=['patient_id', 'side'], how='left')
df_t1_test_ext = df_t1_test_ext.merge(df_asym[['patient_id', 'side'] + TOP_ASYM],
                                      on=['patient_id', 'side'], how='left')

# Fill NaN
df_t1_train_ext[NEW_FEAT_COLS + TOP_ASYM] = df_t1_train_ext[NEW_FEAT_COLS + TOP_ASYM].fillna(0)
df_t1_test_ext[NEW_FEAT_COLS + TOP_ASYM] = df_t1_test_ext[NEW_FEAT_COLS + TOP_ASYM].fillna(0)

print(f'Extended train shape: {df_t1_train_ext.shape}')
print(f'Extended test shape: {df_t1_test_ext.shape}')

# Define feature groups
BASE_ALL = FEAT_COLS_BASE  # 63-68 features
EXT_ALL = FEAT_COLS_BASE + NEW_FEAT_COLS + TOP_ASYM  # Base + new clinical + asymmetry

# View-aware groups (for per-item selection)
VA_SAGITTAL = SAG_COLS + ALL_COLS + META_COLS  # for sagittal items
VA_CORONAL = COR_COLS + ALL_COLS + META_COLS   # for coronal items
VA_SAG_EXT = VA_SAGITTAL + NEW_FEAT_COLS + TOP_ASYM  # + new features
VA_COR_EXT = VA_CORONAL + [c for c in NEW_FEAT_COLS if 'pelvic' in c] + TOP_ASYM  # coronal-relevant new

# Item classification
SAGITTAL_ITEMS = [1, 2, 3, 6, 7, 8, 9, 10, 11, 12, 16, 17]
CORONAL_ITEMS = [4, 5, 13, 14, 15]

print(f'\nFeature group sizes:')
print(f'  Base all: {len(BASE_ALL)}')
print(f'  Extended all: {len(EXT_ALL)}')
print(f'  VA sagittal: {len(VA_SAGITTAL)} (Extended: {len(VA_SAG_EXT)})')
print(f'  VA coronal: {len(VA_CORONAL)} (Extended: {len(VA_COR_EXT)})')

cell_end('Cell 5: Merge Features')


In [ ]:
# === Cell 6: LOSO CV — 4 Strategies × 3 Models ===
cell_start('Cell 6: LOSO CV')

train_pids = sorted(df_t1_train_ext['patient_id'].unique())
print(f'Train patients: {len(train_pids)}')

# Define strategies
STRATEGIES = {
    'base_all': lambda item_i: BASE_ALL,
    'ext_all': lambda item_i: EXT_ALL,
    'base_viewaware': lambda item_i: VA_CORONAL if item_i in CORONAL_ITEMS else VA_SAGITTAL,
    'ext_viewaware': lambda item_i: (VA_COR_EXT if item_i in CORONAL_ITEMS else VA_SAG_EXT),
}

MODELS = ['XGBoost', 'LightGBM'] + (['CatBoost'] if HAVE_CATBOOST else [])

results = {}  # {item_i: {strategy_model: accuracy}}
oof_preds = {}

for item_i in tqdm(range(1, 18), desc='Items'):
    item_name = f'evgs_{item_i}'
    y_all = df_t1_train_ext[item_name].values
    results[item_i] = {}
    oof_preds[item_i] = {}

    for strat_name, feat_fn in STRATEGIES.items():
        feat_cols = [c for c in feat_fn(item_i) if c in df_t1_train_ext.columns]
        X_all = df_t1_train_ext[feat_cols].values.astype(np.float32)
        X_all = np.nan_to_num(X_all, nan=0.0)

        for model_name in MODELS:
            key = f'{strat_name}_{model_name}'
            oof = np.zeros(len(y_all), dtype=np.float32)

            for fold_pid in train_pids:
                mask = df_t1_train_ext['patient_id'].values == fold_pid
                X_tr, y_tr = X_all[~mask], y_all[~mask]
                X_val = X_all[mask]
                spw = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)

                if model_name == 'XGBoost':
                    clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                        scale_pos_weight=spw, use_label_encoder=False,
                        eval_metric='logloss', verbosity=0, random_state=42)
                elif model_name == 'LightGBM':
                    clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                        scale_pos_weight=spw, verbose=-1, random_state=42)
                else:
                    clf = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                        auto_class_weights='Balanced', verbose=0, random_state=42)
                clf.fit(X_tr, y_tr)
                oof[mask] = clf.predict(X_val)

            acc = accuracy_score(y_all, oof)
            results[item_i][key] = acc
            oof_preds[item_i][key] = oof

# Print per-item comparison
print(f'\n{"Item":<6} {"Base_XGB":<8} {"Ext_XGB":<8} {"Base_VA":<8} {"Ext_VA":<8} {"Best":<25} {"BestAcc":<8} {"Delta":<8}')
print('-' * 90)

best_config = {}
for item_i in range(1, 18):
    base_xgb = results[item_i].get('base_all_XGBoost', 0)
    ext_xgb = results[item_i].get('ext_all_XGBoost', 0)
    base_va_key = 'base_viewaware_XGBoost'
    ext_va_key = 'ext_viewaware_XGBoost'
    base_va = results[item_i].get(base_va_key, 0)
    ext_va = results[item_i].get(ext_va_key, 0)

    best_key = max(results[item_i], key=results[item_i].get)
    best_acc = results[item_i][best_key]
    delta = best_acc - base_xgb
    best_config[item_i] = best_key

    print(f'  {item_i:<4} {base_xgb:.4f}  {ext_xgb:.4f}  {base_va:.4f}  {ext_va:.4f}  {best_key:<25} {best_acc:.4f}  {delta:+.4f}')

cell_end('Cell 6: LOSO CV')


In [ ]:
# === Cell 7: Compute S1 & Select Best Config ===
cell_start('Cell 7: Compute S1')

def compute_s1(oof_dict, config_map, df):
    """Compute S1. RMSE is per-patient (L+R combined Total, max=34)."""
    n = len(df)
    all_correct = 0
    all_total = 0
    total_pred = np.zeros(n)
    total_true = np.zeros(n)

    for item_i in range(1, 18):
        key = config_map[item_i]
        pred = oof_dict[item_i][key]
        true = df[f'evgs_{item_i}'].values
        all_correct += np.sum(pred == true)
        all_total += len(true)
        total_pred += pred
        total_true += true

    acc = all_correct / all_total
    pids = df['patient_id'].values
    sides = df['side'].values
    unique_pids = sorted(set(pids))
    rmse_vals = []
    for pid in unique_pids:
        ml = (pids == pid) & (sides == 'left')
        mr = (pids == pid) & (sides == 'right')
        if ml.sum() > 0 and mr.sum() > 0:
            pt = total_pred[ml][0] + total_pred[mr][0]
            tt = total_true[ml][0] + total_true[mr][0]
            rmse_vals.append((pt - tt) ** 2)
    rmse = np.sqrt(np.mean(rmse_vals)) if rmse_vals else 0.0
    s1 = (acc + 1 - rmse / 34.0) / 2
    return acc, rmse, s1

# Config A: Base all XGBoost (single-feature-set baseline)
cfg_base = {i: 'base_all_XGBoost' for i in range(1, 18)}
acc_base, rmse_base, s1_base = compute_s1(oof_preds, cfg_base, df_t1_train_ext)

# Config B: Margin-gated — switch only if margin >= 0.03
cfg_cons = {}
for item_i in range(1, 18):
    base = results[item_i]['base_all_XGBoost']
    best_key = max(results[item_i], key=results[item_i].get)
    best_acc = results[item_i][best_key]
    cfg_cons[item_i] = best_key if (best_acc - base) >= 0.03 else 'base_all_XGBoost'

acc_cons, rmse_cons, s1_cons = compute_s1(oof_preds, cfg_cons, df_t1_train_ext)

# Config C: Per-item best
cfg_best = best_config.copy()
acc_best, rmse_best, s1_best = compute_s1(oof_preds, cfg_best, df_t1_train_ext)

BASELINE_S1_REFERENCE = 0.8097  # all-XGBoost baseline LOSO CV S1; reference for delta tracking

print(f'\nConfig                     Acc      RMSE     S1       Delta vs baseline')
print(f'----------------------------------------------------------------------')
print(f'Base all XGBoost         {acc_base:.4f}   {rmse_base:.4f}   {s1_base:.4f}   {s1_base-BASELINE_S1_REFERENCE:+.4f}')
print(f'Margin-gated       {acc_cons:.4f}   {rmse_cons:.4f}   {s1_cons:.4f}   {s1_cons-BASELINE_S1_REFERENCE:+.4f}')
print(f'Per-item best          {acc_best:.4f}   {rmse_best:.4f}   {s1_best:.4f}   {s1_best-BASELINE_S1_REFERENCE:+.4f}')

# Show conservative switches
print(f'\nMargin-gated switches (margin >= 0.03):')
n_switches = 0
for item_i in range(1, 18):
    if cfg_cons[item_i] != 'base_all_XGBoost':
        base = results[item_i]['base_all_XGBoost']
        new = results[item_i][cfg_cons[item_i]]
        print(f'  Item {item_i}: {cfg_cons[item_i]} ({new:.4f} vs {base:.4f}, +{new-base:.4f})')
        n_switches += 1
if n_switches == 0:
    print('  (none — no item improved by >= 0.03)')

cell_end('Cell 7: Compute S1')

In [ ]:
# === Cell 8: Generate Submissions ===
cell_start('Cell 8: Generate Submissions')

# Use conservative config
SUBMIT_CONFIG = cfg_cons

# Train final models on all training data
test_predictions = {}
for item_i in tqdm(range(1, 18), desc='Final training'):
    item_name = f'evgs_{item_i}'
    cfg = SUBMIT_CONFIG[item_i]
    parts = cfg.rsplit('_', 1)
    model_name = parts[-1]
    strat_name = '_'.join(parts[:-1])

    feat_fn = STRATEGIES[strat_name]
    feat_cols = [c for c in feat_fn(item_i) if c in df_t1_train_ext.columns]

    X_train = df_t1_train_ext[feat_cols].values.astype(np.float32)
    X_train = np.nan_to_num(X_train, nan=0.0)
    y_train = df_t1_train_ext[item_name].values

    X_test = df_t1_test_ext[feat_cols].values.astype(np.float32)
    X_test = np.nan_to_num(X_test, nan=0.0)

    spw = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

    if model_name == 'XGBoost':
        clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
            scale_pos_weight=spw, use_label_encoder=False,
            eval_metric='logloss', verbosity=0, random_state=42)
    elif model_name == 'LightGBM':
        clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
            scale_pos_weight=spw, verbose=-1, random_state=42)
    else:
        clf = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
            auto_class_weights='Balanced', verbose=0, random_state=42)

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    test_predictions[item_i] = {}
    for idx in range(len(df_t1_test_ext)):
        row = df_t1_test_ext.iloc[idx]
        test_predictions[item_i][(row['patient_id'], row['side'])] = int(preds[idx])

# Track 2 baseline placeholder predictions for the submission CSV schema.
# These rows are overwritten by the Track-2 clinical classifier in notebook 03;
# kept here so the standalone xgb_baseline.csv has the complete Track-2
# row format expected by downstream tooling.
TRACK2_BASELINE_PLACEHOLDER = {
    4: ('type3', 'type3'), 6: ('type2', 'type3'), 7: ('type3', 'type3'),
    13: ('WNL', 'WNL'), 26: ('WNL', 'WNL'), 35: ('type2', 'type2'),
    39: ('type1', 'type1'), 42: ('type2', 'type2'), 50: ('type2', 'type2'),
}

# Build CSV
rows = []
for pid in sorted(T1_TEST_IDS):
    row = {'ID': f'track1-{pid}'}
    total = 0
    for side, prefix in [('left', 'L'), ('right', 'R')]:
        for i in range(1, 18):
            val = test_predictions.get(i, {}).get((pid, side), 0)
            row[f'{prefix}{i}'] = val
            total += val
    row['Total'] = total
    row['Left_gait_subtype'] = -1
    row['Right_gait_subtype'] = -1
    rows.append(row)

for pid in sorted(T2_TEST_IDS):
    row = {'ID': f'track2-{pid}'}
    for p in ['L', 'R']:
        for i in range(1, 18):
            row[f'{p}{i}'] = -1
    row['Total'] = -1
    lt, rt = TRACK2_BASELINE_PLACEHOLDER.get(pid, ('WNL', 'WNL'))
    row['Left_gait_subtype'] = lt
    row['Right_gait_subtype'] = rt
    rows.append(row)

cols = ['ID'] + [f'{p}{i}' for p in ['L','R'] for i in range(1,18)] + ['Total', 'Left_gait_subtype', 'Right_gait_subtype']
sub_df = pd.DataFrame(rows)[cols]

# (Intermediate cfg_cons output is held in `sub_df` for Cell 8b to apply
# the items 4 + 13 view-aware overrides; Cell 8c then writes the single
# canonical xgb_baseline.csv consumed by the rest of the pipeline.)

print(f'Margin-gated cfg_cons output prepared in memory; shape: {sub_df.shape}')

cell_end('Cell 8: Generate Submissions')


In [ ]:
# === Cell 8b: items 4 + 13 view-aware override ===
# Items 4 (HindfootVarusValgus) and 13 (HipFlexExtSwing) are coronal-plane
# phenomena that benefit from view-aware coronal feature subsets. The
# conservative LOSO selection threshold (margin >= 0.03) is calibrated for
# the broader item set; for these two coronal items we apply the view-aware
# variant directly based on biomechanical reasoning.
#
# Strategy: train one XGBoost per item using base_viewaware features (coronal
# subset = cor_* + all_* + n_*), generate test predictions, replace L4/R4
# and L13/R13 cells in sub_df.
cell_start('Cell 8b: items 4 + 13 view-aware override')

OVERRIDE_ITEMS = [4, 13]

for item_i in OVERRIDE_ITEMS:
    item_name = f'evgs_{item_i}'
    feat_cols = [c for c in VA_CORONAL if c in df_t1_train_ext.columns]
    X_train = np.nan_to_num(df_t1_train_ext[feat_cols].values.astype(np.float32), nan=0.0)
    y_train = df_t1_train_ext[item_name].values
    X_test = np.nan_to_num(df_t1_test_ext[feat_cols].values.astype(np.float32), nan=0.0)
    spw = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

    clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
        scale_pos_weight=spw, use_label_encoder=False,
        eval_metric='logloss', verbosity=0, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    # Build a (pid, side) -> pred map and overlay onto sub_df
    pred_map = {}
    for idx in range(len(df_t1_test_ext)):
        row = df_t1_test_ext.iloc[idx]
        pred_map[(int(row['patient_id']), row['side'])] = int(preds[idx])

    for pid in T1_TEST_IDS:
        for side, prefix in [('left', 'L'), ('right', 'R')]:
            new_val = pred_map.get((pid, side))
            if new_val is None:
                continue
            col = f'{prefix}{item_i}'
            sub_df.loc[sub_df['ID'] == f'track1-{pid}', col] = new_val
    print(f'  overrode item {item_i} (L{item_i}/R{item_i}) with base_viewaware_XGBoost predictions')

# Recompute Total for Track-1 rows after override
ITEM_COLS_RECOMP = [f'{p}{i}' for p in ['L', 'R'] for i in range(1, 18)]
t1_mask = sub_df['ID'].str.startswith('track1-')
sub_df.loc[t1_mask, 'Total'] = sub_df.loc[t1_mask, ITEM_COLS_RECOMP].astype(int).sum(axis=1)
print(f'recomputed Total after override')

cell_end('Cell 8b: items 4 + 13 view-aware override')

In [ ]:
# === Cell 8c: emit canonical artifacts for the assembly pipeline ===
# Two outputs, both required by downstream stages:
# - xgb_baseline.csv: read by src/cv4chl/ml_merge.py as the runtime base table
# - reference_baseline.csv: read by notebooks 04-07 as the table they overlay onto
#
# Both files are written to {SUBMIT_DIR} = {ROOT}/5_outputs/submissions/, where
# ROOT is resolved from CV4CHL_REPO_ROOT (set by reproduce.py) or the standalone
# Colab default.
cell_start('Cell 8c: emit canonical artifacts')

xgb_baseline_path = f'{SUBMIT_DIR}/xgb_baseline.csv'
reference_baseline_path = f'{SUBMIT_DIR}/reference_baseline.csv'

sub_df.to_csv(xgb_baseline_path, index=False)
sub_df.to_csv(reference_baseline_path, index=False)

print(f'  wrote {xgb_baseline_path}')
print(f'  wrote {reference_baseline_path}')
print(f'  shape: {sub_df.shape}')

cell_end('Cell 8c: emit canonical artifacts')


In [ ]:
# === Cell 9: Summary ===
cell_start('Summary')

total_time = sum(t for t in _cell_times.values() if t is not None)
time_report = '\n'.join(
    f'  {name}: {t:.1f}s ({t/60:.1f}min)'
    for name, t in _cell_times.items() if t is not None
)

# Margin-gated switch summary
switch_summary = []
for item_i in range(1, 18):
    if cfg_cons[item_i] != 'base_all_XGBoost':
        switch_summary.append(f'Item {item_i}: {cfg_cons[item_i]}')

print(f"""
{'='*70}
  Notebook 02 (XGB baseline + view-aware features + clinical features) — Summary
{'='*70}

Execution time:
{time_report}
  ----------
  Total: {total_time:.1f}s ({total_time/60:.1f}min)

Environment:
  Platform: {'Colab' if IN_COLAB else 'Local'}

New Features Added:
  Clinical: {NEW_FEAT_COLS}
  Asymmetry: {TOP_ASYM}
  Total new: {len(NEW_FEAT_COLS) + len(TOP_ASYM)}

Feature Groups:
  Base all: {len(BASE_ALL)}
  Extended all: {len(EXT_ALL)}
  VA sagittal: {len(VA_SAGITTAL)} -> Extended: {len(VA_SAG_EXT)}
  VA coronal: {len(VA_CORONAL)} -> Extended: {len(VA_COR_EXT)}

Track 1 LOSO CV Results:
  Config                     Acc       RMSE       S1      Delta vs baseline
  -----------------------------------------------------------------------
  Base all XGBoost         {acc_base:.4f}    {rmse_base:.4f}    {s1_base:.4f}    {s1_base-BASELINE_S1_REFERENCE:+.4f}
  Margin-gated       {acc_cons:.4f}    {rmse_cons:.4f}    {s1_cons:.4f}    {s1_cons-BASELINE_S1_REFERENCE:+.4f}
  Per-item best          {acc_best:.4f}    {rmse_best:.4f}    {s1_best:.4f}    {s1_best-BASELINE_S1_REFERENCE:+.4f}

Margin-gated switches: {len(switch_summary)}
  {chr(10).join(switch_summary) if switch_summary else '(none)'}

Track 2: handled by notebook 03 (clinical classifier)

Outputs:
  5_outputs/submissions/xgb_baseline.csv         (canonical Track-1 baseline)
  5_outputs/submissions/reference_baseline.csv   (same content; consumed by notebooks 04-07 as overlay base)
{'='*70}
""")

cell_end('Summary')